In [9]:
# ==============================
# YFINANCE CLEAN EXPLORATION
# ==============================

import yfinance as yf
import pandas as pd

pd.set_option('display.max_columns', None)

ticker = yf.Ticker("INFY.NS")

# ------------------------------
# BASIC INFO
# ------------------------------

print("\n=== INFO ===")
try:
    info = ticker.info or {}
    print("Fields:", len(info))
    info_df = pd.DataFrame(list(info.items()), columns=["key", "value"])
    print(info_df.head(10))
except Exception as e:
    print("Info error:", e)

# ------------------------------
# PRICE DATA
# ------------------------------

print("\n=== PRICE DATA ===")
try:
    hist = ticker.history(period="1y")
    print(hist.head())
    print("Shape:", hist.shape)
except Exception as e:
    print("history: error", e)

# ------------------------------
# ACTIONS
# ------------------------------

try:
    print("\n=== ACTIONS ===")
    print(ticker.actions.head())
except Exception as e:
    print("actions: error", e)

# ------------------------------
# FINANCIALS
# ------------------------------

print("\n=== FINANCIALS ===")
try:
    print(ticker.financials.head())
except Exception:
    print("No financials")

# ------------------------------
# NEWS (SAFE VERSION)
# ------------------------------

print("\n=== NEWS ===")
try:
    news = getattr(ticker, 'news', None)
    if news:
        print("Total news:", len(news))
        first = news[0]
        for k in first:
            print(f"{k}: {first[k]}")
    else:
        print("No news data")
except Exception as e:
    print("News error:", e)

# ------------------------------
# SUMMARY
# ------------------------------

try:
    print("\n=== SUMMARY ===")
    print("Price rows:", getattr(hist, 'shape', (0,))[0])
except Exception:
    print("No history available for summary")


=== INFO ===
Fields: 172
           key                                 value
0     address1                      Plot No. 44/97 A
1     address2  3rd cross Electronic City Hosur Road
2         city                             Bengaluru
3          zip                               56 0100
4      country                                 India
5        phone                       91 80 2852 0261
6          fax                       91 80 2852 0362
7      website               https://www.infosys.com
8     industry       Information Technology Services
9  industryKey       information-technology-services

=== PRICE DATA ===
                                  Open         High          Low        Close  \
Date                                                                            
2025-04-17 00:00:00+05:30  1357.816353  1388.702352  1338.682651  1378.698364   
2025-04-21 00:00:00+05:30  1369.471458  1429.495148  1369.471458  1409.292969   
2025-04-22 00:00:00+05:30  1398.609126  1398.60

In [3]:
import os
import requests
from datetime import datetime, timedelta
import pandas as pd
# Explore Polygon and Tiingo APIs (new cell)


# Use API keys from environment variables
POLYGON_API_KEY = os.getenv("POLYGON_API_KEY")
TIINGO_API_KEY = os.getenv("TIINGO_API_KEY")

def polygon_get_aggregates(symbol, from_date, to_date, multiplier=1, timespan='day'):
    """Fetch aggregates (ohlcv) from Polygon."""
    if not POLYGON_API_KEY:
        print("POLYGON_API_KEY not set")
        return None
    url = f"https://api.polygon.io/v2/aggs/ticker/{symbol}/range/{multiplier}/{timespan}/{from_date}/{to_date}"
    params = {"adjusted": "true", "sort": "asc", "limit": 50000, "apiKey": POLYGON_API_KEY}
    r = requests.get(url, params=params, timeout=20)
    r.raise_for_status()
    j = r.json()
    results = j.get("results", [])
    if not results:
        print("No aggregate results from Polygon")
        return None
    df = pd.DataFrame(results)
    # t is epoch ms
    df["timestamp"] = pd.to_datetime(df["t"], unit="ms")
    df = df.rename(columns={"o": "open", "h": "high", "l": "low", "c": "close", "v": "volume"})
    df = df[["timestamp", "open", "high", "low", "close", "volume"]]
    df = df.set_index("timestamp")
    print("Polygon aggregates shape:", df.shape)
    display(df.head())
    return df

def polygon_ticker_meta(symbol):
    """Fetch ticker metadata from Polygon (v3 reference)."""
    if not POLYGON_API_KEY:
        print("POLYGON_API_KEY not set")
        return None
    url = f"https://api.polygon.io/v3/reference/tickers/{symbol}"
    params = {"apiKey": POLYGON_API_KEY}
    r = requests.get(url, params=params, timeout=10)
    if r.status_code == 404:
        print("Ticker not found in Polygon:", symbol)
        return None
    r.raise_for_status()
    j = r.json().get("results", {})
    print("Polygon ticker metadata keys:", list(j.keys())[:20])
    display(pd.Series(j).to_frame(name="value").head(20))
    return j

def tiingo_get_prices(symbol, start_date, end_date):
    """Fetch historical prices from Tiingo daily endpoint."""
    if not TIINGO_API_KEY:
        print("TIINGO_API_KEY not set")
        return None
    url = f"https://api.tiingo.com/tiingo/daily/{symbol}/prices"
    params = {"startDate": start_date, "endDate": end_date}
    headers = {"Content-Type": "application/json", "Authorization": f"Token {TIINGO_API_KEY}"}
    r = requests.get(url, params=params, headers=headers, timeout=20)
    if r.status_code == 404:
        print("Ticker not found in Tiingo:", symbol)
        return None
    r.raise_for_status()
    data = r.json()
    if not data:
        print("No price data from Tiingo")
        return None
    df = pd.DataFrame(data)
    if "date" in df.columns:
        df["date"] = pd.to_datetime(df["date"]).dt.tz_convert(None)
        df = df.set_index("date")
    print("Tiingo prices shape:", df.shape)
    display(df.head())
    return df

# Example usage (adjust symbols as needed)
end = datetime.utcnow().date()
start = end - timedelta(days=365)
# polygon symbols are usually without exchange suffix; tiingo expects its symbol format
example_symbol_polygon = "AAPL"       # change to a symbol you expect Polygon to have
example_symbol_tiingo = "AAPL"       # change as needed

# Run queries if keys present
if POLYGON_API_KEY:
    try:
        polygon_ticker_meta(example_symbol_polygon)
        polygon_get_aggregates(example_symbol_polygon, start.isoformat(), end.isoformat())
    except Exception as e:
        print("Polygon error:", e)

if TIINGO_API_KEY:
    try:
        tiingo_get_prices(example_symbol_tiingo, start.isoformat(), end.isoformat())
    except Exception as e:
        print("Tiingo error:", e)

# Notes:
# - Set POLYGON_API_KEY and TIINGO_API_KEY in environment before running.
# - Adjust symbols (e.g., add exchange suffixes) depending on coverage for your tickers.

Polygon ticker metadata keys: ['ticker', 'name', 'market', 'locale', 'primary_exchange', 'type', 'active', 'currency_name', 'cik', 'composite_figi', 'share_class_figi', 'market_cap', 'phone_number', 'address', 'description', 'sic_code', 'sic_description', 'ticker_root', 'homepage_url', 'total_employees']


,value
ticker,AAPL
name,Apple Inc.
market,stocks
locale,us
primary_exchange,XNAS
type,CS
active,True
currency_name,usd
cik,0000320193
composite_figi,BBG000B9XRY4


Polygon aggregates shape: (250, 5)


,open,high,low,close,volume
timestamp,,,,,
2025-04-21 04:00:00,193.265,193.8000,189.8112,193.16,46742537.0
2025-04-22 04:00:00,196.120,201.5900,195.9700,199.74,52976371.0
2025-04-23 04:00:00,206.000,208.0000,202.7990,204.60,52929165.0
2025-04-24 04:00:00,204.890,208.8299,202.9400,208.37,47310989.0
2025-04-25 04:00:00,206.365,209.7500,206.2000,209.28,38222258.0


Tiingo prices shape: (250, 12)


,close,high,low,open,volume,adjClose,adjHigh,adjLow,adjOpen,adjVolume,divCash,splitFactor
date,,,,,,,,,,,,
2025-04-21,193.16,193.8000,189.8112,193.265,46742537,192.333627,192.970889,188.999154,192.438178,46742537,0.0,1.0
2025-04-22,199.74,201.5900,195.9700,196.120,52976371,198.885476,200.727562,195.131605,195.280963,52976371,0.0,1.0
2025-04-23,204.60,208.0000,202.7990,206.000,52929165,203.724684,207.110139,201.931389,205.118695,52929165,0.0,1.0
2025-04-24,208.37,208.8299,202.9400,204.890,47310989,207.478556,207.936488,202.071786,204.013444,47310989,0.0,1.0
2025-04-25,209.28,209.7500,206.2000,206.365,38222258,208.384663,208.852652,205.317839,205.482133,38222258,0.0,1.0


In [ ]:
# python
import requests
import json

API_KEY = "RspQojXxp7ZMvxGtoGi3nkKXAFO14lqJ"
BASE_URL = "https://api.polygon.io"

def explore_endpoint(path, params=None):
    url = f"{BASE_URL}{path}"
    if params is None:
        params = {}
    params["apiKey"] = API_KEY    # use defined variable and correct param name
    print(f"exploring: {path}")
    res = requests.get(url, params=params, timeout=15)
    print("status:", res.status_code)
    try:
        data = res.json()
        print("keys:", list(data.keys()))
        print(json.dumps(data, indent=2)[:1000])
    except Exception as e:
        print("Error parsing JSON:", e)

explore_endpoint("/v3/reference/tickers/AAPL", {"limit": 3})
explore_endpoint("/v2/aggs/ticker/AAPL/range/1/day/2023-01-01/2023-12-31")
explore_endpoint("/v2/reference/news", {"limit": 3})
# python
import requests
import json

API_KEY = "RspQojXxp7ZMvxGtoGi3nkKXAFO14lqJ"
BASE_URL = "https://api.polygon.io"

def explore_endpoint(path, params=None):
    url = f"{BASE_URL}{path}"
    if params is None:
        params = {}
    params["apiKey"] = API_KEY    # use defined variable and correct param name
    print(f"exploring: {path}")
    res = requests.get(url, params=params, timeout=15)
    print("status:", res.status_code)
    try:
        data = res.json()
        print("keys:", list(data.keys()))
        print(json.dumps(data, indent=2)[:1000])
    except Exception as e:
        print("Error parsing JSON:", e)

explore_endpoint("/v3/reference/tickers/AAPL", {"limit": 3})
explore_endpoint("/v2/aggs/ticker/AAPL/range/1/day/2023-01-01/2023-12-31")
explore_endpoint("/v2/reference/news", {"limit": 3})

SyntaxError: invalid syntax (597138803.py, line 26)